In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

path = "./files/topics/"
outpath = "./files/ckip-topics/"
paths=[]
for dirPath, dirNames, fileNames in os.walk(path):
    for f in fileNames:
        paths.append(os.path.join(dirPath, f))
print(len(paths))

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
pipeline = CkipPipeline(opts={'con_parser': {'username': "infoamplify", 'password': "2223pj[DB]"}})

In [ ]:
from multiprocessing import Pool
import json
from tqdm import tqdm
from pathlib import Path

def work(path):
    f=open(path,mode="r",encoding='utf-8')        
    jjson = json.loads(f.read())
    ws=[]
    pos=[]
    for info in jjson["content"]:
        doc = CkipDocument(raw=info)
        pipeline.get_ws(doc)
        pipeline.get_pos(doc)
        ws.append(flatten(doc.ws.to_list()))
        pos.append(flatten(doc.pos.to_list()))
    content=[]
    for i in range(len(ws)):
        sentence=[]
        for j in range(len(ws[i])):
            sentence.append({"word":ws[i][j],"tag":pos[i][j]})
        content.append(sentence)
    f.close()
    
    directory=Path(outpath+path).parent.absolute()
    if not os.path.exists(directory):
        os.makedirs(directory)
        
    out_file = open(outpath+path,mode="w",encoding='utf-8')
    json.dump({"content":content}, out_file,ensure_ascii=False)
    
    out_file.close()
    #print(" Process %s waiting %s seconds" % (work_data[0], work_data[1]))
    #print(" Process %s Finished." % work_data[0])
    

def pool_handler():
    with Pool(24) as p:
        p.map(work, paths)


if __name__ == '__main__':
    pool_handler()